# 09 存活分析：發病後，誰的預後比較差？

松柏護理之家 121 位感染住民中，19 人死亡。
主治醫師問：「哪些人的死亡風險比較高？能不能量化？」

流程：**建立分析資料集 → KM 全體曲線 → 分組比較 → Log-rank 檢定 → Cox 迴歸 → HR 森林圖**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: 建立存活分析資料集 ---
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["death_date"] = pd.to_datetime(df["death_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# 限定感染住民
cases = df[df["infected"] == 1].copy()

# 事件指標：1=死亡, 0=存活（右設限）
cases["event"] = (cases["outcome"] == "dead").astype(int)

# 存活時間
# 死亡者：time = death_date - symptom_onset_date
# 存活者：time = investigation_end - symptom_onset_date（設限）
investigation_end = cases["symptom_onset_date"].max() + pd.Timedelta(days=14)
cases["end_date"] = cases.apply(
    lambda r: r["death_date"] if r["event"] == 1 else investigation_end, axis=1
)
cases["time_to_event"] = (cases["end_date"] - cases["symptom_onset_date"]).dt.days

print(f"感染住民：{len(cases)}")
print(f"死亡：{cases['event'].sum()}")
print(f"存活（設限）：{(cases['event'] == 0).sum()}")
print(f"\n調查結束日：{investigation_end.date()}")
print(f"\n死亡者存活時間（天）：")
died = cases[cases["event"] == 1]
print(f"  mean = {died['time_to_event'].mean():.1f}")
print(f"  median = {died['time_to_event'].median():.1f}")
print(f"  range = {died['time_to_event'].min()} \u2013 {died['time_to_event'].max()} 天")

In [ ]:
# --- Step 2: Kaplan-Meier 全體存活曲線 ---
from lifelines import KaplanMeierFitter

kmf = KaplanMeierFitter()
kmf.fit(cases["time_to_event"], event_observed=cases["event"],
        label="全體感染住民")

fig, ax = plt.subplots(figsize=(8, 5))
kmf.plot_survival_function(ax=ax)
ax.set_title("Kaplan-Meier 存活曲線（全體感染住民）")
ax.set_xlabel("發病後天數")
ax.set_ylabel("存活機率")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

# 中位數存活時間
median_surv = kmf.median_survival_time_
print(f"中位存活時間：{median_surv}")
print("\u2192 如果中位數顯示 inf，代表超過 50% 的人在觀察期內存活（這是好消息）")

In [ ]:
# --- Step 3: 按嚴重度分組的存活曲線 ---
severity_levels = ["mild", "moderate", "severe"]
colors = {"mild": "#41b6c4", "moderate": "#fed976", "severe": "#e31a1c"}

fig, ax = plt.subplots(figsize=(8, 5))

for sev in severity_levels:
    mask = cases["clinical_severity"] == sev
    sub = cases[mask]
    if len(sub) == 0:
        continue
    kmf_sev = KaplanMeierFitter()
    kmf_sev.fit(sub["time_to_event"], event_observed=sub["event"],
                label=f"{sev} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf_sev.plot_survival_function(ax=ax, color=colors[sev])

ax.set_title("存活曲線（按嚴重度分組）")
ax.set_xlabel("發病後天數")
ax.set_ylabel("存活機率")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

print("\u2192 觀察：severe 組的存活曲線是否明顯低於 mild/moderate？")
print("\u2192 曲線分離越早、距離越大 = 嚴重度對存活的影響越強")

In [ ]:
# --- Step 4: Log-rank 檢定 ---
from lifelines.statistics import logrank_test

# 比較 severe vs non-severe
severe = cases[cases["clinical_severity"] == "severe"]
non_severe = cases[cases["clinical_severity"].isin(["mild", "moderate"])]

result = logrank_test(
    severe["time_to_event"], non_severe["time_to_event"],
    event_observed_A=severe["event"],
    event_observed_B=non_severe["event"],
)

print("=== Log-rank 檢定：severe vs non-severe ===")
print(f"  test statistic = {result.test_statistic:.3f}")
print(f"  p-value = {result.p_value:.4f}")

if result.p_value < 0.05:
    print("  \u2192 p < 0.05，兩組存活曲線有統計顯著差異")
else:
    print("  \u2192 p \u2265 0.05，無法拒絕兩組存活曲線相同的虛無假設")

# 也比較 COPD vs no COPD
copd_yes = cases[cases["comorbidity_copd"] == 1]
copd_no = cases[cases["comorbidity_copd"] == 0]

result_copd = logrank_test(
    copd_yes["time_to_event"], copd_no["time_to_event"],
    event_observed_A=copd_yes["event"],
    event_observed_B=copd_no["event"],
)

print(f"\n=== Log-rank 檢定：COPD vs no COPD ===")
print(f"  test statistic = {result_copd.test_statistic:.3f}")
print(f"  p-value = {result_copd.p_value:.4f}")

In [ ]:
# --- Step 5: Cox 比例風險迴歸 ---
from lifelines import CoxPHFitter

# 建立 Cox 迴歸資料集
cox_df = cases[[
    "time_to_event", "event", "age", "sex",
    "comorbidity_copd", "comorbidity_chf", "comorbidity_dm",
    "comorbidity_cancer", "immunosuppressed",
]].copy()
cox_df["is_male"] = (cox_df["sex"] == "M").astype(int)
cox_df = cox_df.drop(columns=["sex"])

# 配適模型
cph = CoxPHFitter()
cph.fit(cox_df, duration_col="time_to_event", event_col="event")

print("=== Cox 比例風險迴歸結果 ===")
cph.print_summary()

# 簡潔的 HR 表格
print("\n=== Hazard Ratio 摘要 ===")
summary = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
summary.columns = ["HR", "HR_lower", "HR_upper", "p_value"]
print(summary.round(3).to_string())

print("\n\u2192 HR > 1 代表死亡風險較高（危險因子）")
print("\u2192 HR < 1 代表死亡風險較低（保護因子）")
print("\u2192 95% CI 包含 1 則不顯著")

In [ ]:
# --- Step 6: HR 森林圖 ---
fig, ax = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
ax.set_title("Cox Regression \u2014 Hazard Ratio（log scale）")
plt.tight_layout()
plt.show()

print("\u2192 森林圖中的點 = log(HR)，誤差線 = 95% CI")
print("\u2192 點在虛線右邊 = HR > 1（危險因子）")
print("\u2192 誤差線跨越虛線 = 不顯著")

In [ ]:
# --- 補充：COPD 分組 Kaplan-Meier ---
fig, ax = plt.subplots(figsize=(8, 5))

for label, mask in [("COPD", cases["comorbidity_copd"] == 1),
                     ("No COPD", cases["comorbidity_copd"] == 0)]:
    sub = cases[mask]
    kmf_sub = KaplanMeierFitter()
    kmf_sub.fit(sub["time_to_event"], event_observed=sub["event"],
                label=f"{label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf_sub.plot_survival_function(ax=ax)

ax.set_title("存活曲線：COPD vs No COPD")
ax.set_xlabel("發病後天數")
ax.set_ylabel("存活機率")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

print(f"\nCOPD Log-rank p-value = {result_copd.p_value:.4f}")

## 小結

| 步驟 | 學到的技能 |
|------|------------|
| 建立資料集 | 計算 `time_to_event` 與 `event` 指標，正確處理設限 |
| KM 曲線 | `KaplanMeierFitter` 全體 + 分組存活曲線 |
| Log-rank | `logrank_test()` 比較兩組存活曲線差異 |
| Cox 迴歸 | `CoxPHFitter` 多因子分析，解讀 HR |
| 森林圖 | `cph.plot()` 視覺化各因子的 HR |

**結論**：存活分析比單純的致死率（CFR）更精確——它同時考慮「有沒有死亡」和「多快死亡」。
Cox 迴歸可以同時調整多個因子，找出獨立的預後危險因子。

下一章（Ch10），我們嘗試用全部特徵訓練機器學習模型 → 預測感染與重症。